<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/main/RA3_LAB4/EXPERIENCIA_5%20/RA3_Lab_N%C2%B04_EXP5_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/alex/RA3_LAB4/EXPERIENCIA_5/RA3_Lab_N%C2%B04_EXP5_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiencia 5 - Programación de Turnos con Restricciones Duras y Blandas (DEAP)

**IC415 - Inteligencia Computacional - RA3 / Laboratorio N 4**

Se resuelve el problema de programación de enfermeras (NSP): asignar 8 enfermeras a los 21 turnos
de una semana (7 días × 3 turnos) satisfaciendo restricciones duras (no trabajar turnos
consecutivos, tope de 5 turnos por semana y cobertura por turno) y maximizando la satisfacción de
las preferencias individuales (restricciones blandas). Se parte de la implementación orientativa
basada en penalización, se identifican sus límites y se implementa y compara una mejora por
reparación de individuos.

El contenido del notebook es deliberadamente conciso: el código y las figuras son la evidencia
técnica reproducible; la interpretación extensa corresponde al informe.

## 0. Configuración del entorno

In [ ]:
import importlib, subprocess, sys
if importlib.util.find_spec("deap") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deap"], check=False)

import os, time, random, zipfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from deap import base, creator, tools, algorithms

SEMILLA_BASE = 42
random.seed(SEMILLA_BASE); np.random.seed(SEMILLA_BASE)
FIGS_DIR = "figuras_informe_exp5"; os.makedirs(FIGS_DIR, exist_ok=True)
print("Figuras en:", os.path.abspath(FIGS_DIR))

Figuras en: /tmp/figuras_informe_exp5


## 1. Modelo del problema y representación

Siguiendo la implementación orientativa, cada solución se representa como una **lista binaria** de
8×21 = 168 posiciones: los 21 turnos de la semana (7 días × 3 turnos) de cada una de las 8
enfermeras concatenados. Un 1 indica que la enfermera trabaja ese turno. La clase encapsula las
restricciones y devuelve, además del costo penalizado, el desglose de violaciones para poder
verificar la factibilidad (cero violaciones duras).

In [ ]:
class NurseSchedulingProblem:
    """Encapsula el NSP: restricciones duras/blandas y costo penalizado."""
    def __init__(self, hardConstraintPenalty):
        self.hardConstraintPenalty = hardConstraintPenalty
        self.nurses = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
        # preferencias (mañana, tarde, noche): 1 = turno aceptado por la enfermera
        self.shiftPreference = [[1,0,0],[1,1,0],[0,0,1],[0,1,0],[0,0,1],[1,1,1],[0,1,1],[1,1,1]]
        self.shiftMin = [2, 2, 1]; self.shiftMax = [3, 4, 2]   # cobertura por turno
        self.maxShiftsPerWeek = 5; self.weeks = 1
        self.shiftPerDay = len(self.shiftMin); self.shiftsPerWeek = 7 * self.shiftPerDay
    def __len__(self): return len(self.nurses) * self.shiftsPerWeek * self.weeks
    def getNurseShifts(self, sch):
        spn = self.__len__() // len(self.nurses); d = {}; k = 0
        for n in self.nurses: d[n] = sch[k:k+spn]; k += spn
        return d
    def countConsecutiveShiftViolations(self, d):
        v = 0
        for s in d.values():
            for a, b in zip(s, s[1:]):
                if a == 1 and b == 1: v += 1
        return v
    def countShiftsPerWeekViolations(self, d):
        v = 0; lst = []
        for s in d.values():
            for i in range(0, self.weeks*self.shiftsPerWeek, self.shiftsPerWeek):
                w = sum(s[i:i+self.shiftsPerWeek]); lst.append(w)
                if w > self.maxShiftsPerWeek: v += w - self.maxShiftsPerWeek
        return lst, v
    def countNursesPerShiftViolations(self, d):
        tot = [sum(sh) for sh in zip(*d.values())]; v = 0
        for i, num in enumerate(tot):
            j = i % self.shiftPerDay
            if num > self.shiftMax[j]: v += num - self.shiftMax[j]
            elif num < self.shiftMin[j]: v += self.shiftMin[j] - num
        return tot, v
    def countShiftPreferenceViolations(self, d):
        v = 0
        for idx, pref in enumerate(self.shiftPreference):
            p = pref * (self.shiftsPerWeek // self.shiftPerDay)
            for pr, sh in zip(p, d[self.nurses[idx]]):
                if pr == 0 and sh == 1: v += 1
        return v
    def hardViolations(self, sch):
        d = self.getNurseShifts(sch)
        c = self.countConsecutiveShiftViolations(d)
        w = self.countShiftsPerWeekViolations(d)[1]
        n = self.countNursesPerShiftViolations(d)[1]
        return c, w, n, c + w + n
    def softViolations(self, sch): return self.countShiftPreferenceViolations(self.getNurseShifts(sch))
    def getCost(self, sch):
        c, w, n, hard = self.hardViolations(sch)
        return self.hardConstraintPenalty * hard + self.softViolations(sch)

# Preferencias documentadas
_nsp = NurseSchedulingProblem(5)
tabla_pref = pd.DataFrame(_nsp.shiftPreference, index=_nsp.nurses, columns=["Mañana", "Tarde", "Noche"])
print("Preferencias por enfermera (1 = turno aceptado):"); print(tabla_pref)
print("\nCobertura requerida  min:", _nsp.shiftMin, " max:", _nsp.shiftMax,
      "| máx turnos/semana:", _nsp.maxShiftsPerWeek)

Preferencias por enfermera (1 = turno aceptado):
   Mañana  Tarde  Noche
A       1      0      0
B       1      1      0
C       0      0      1
D       0      1      0
E       0      0      1
F       1      1      1
G       0      1      1
H       1      1      1

Cobertura requerida  min: [2, 2, 1]  max: [3, 4, 2] | máx turnos/semana: 5


## 2. Motor evolutivo con elitismo

Se emplea el algoritmo genético con elitismo del material orientativo (el salón de la fama se
inyecta intacto en la siguiente generación). El mismo motor sirve para la línea base por
penalización y para la variante con reparación, activando en este último caso la reparación de
cada individuo antes de evaluarlo.

In [ ]:
def eaElitism(pop, tb, cxpb, mutpb, ngen, hof, repair=None, nsp=None, stats=None):
    logbook = tools.Logbook(); logbook.header = ['gen', 'min', 'avg']
    for ind in pop:
        if repair: repair(nsp, ind)
        ind.fitness.values = tb.evaluate(ind)
    hof.update(pop); hof_size = len(hof.items)
    logbook.record(gen=0, **(stats.compile(pop) if stats else {}))
    for gen in range(1, ngen + 1):
        off = tb.select(pop, len(pop) - hof_size)
        off = algorithms.varAnd(off, tb, cxpb, mutpb)
        for ind in off:
            if repair: repair(nsp, ind)
            if (not ind.fitness.valid) or repair:
                ind.fitness.values = tb.evaluate(ind)
        off.extend(hof.items); hof.update(off); pop[:] = off
        logbook.record(gen=gen, **(stats.compile(pop) if stats else {}))
    return pop, logbook

if not hasattr(creator, "FitMin"):
    creator.create("FitMin", base.Fitness, weights=(-1.0,))
    creator.create("Indiv", list, fitness=creator.FitMin)

def construir_toolbox(nsp):
    tb = base.Toolbox()
    tb.register("bit", random.randint, 0, 1)
    tb.register("individual", tools.initRepeat, creator.Indiv, tb.bit, len(nsp))
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("evaluate", lambda ind: (nsp.getCost(ind),))
    tb.register("select", tools.selTournament, tournsize=2)
    tb.register("mate", tools.cxTwoPoint)
    tb.register("mutate", tools.mutFlipBit, indpb=1.0/len(nsp))
    return tb

def correr(penalty, seed, pop=300, gen=200, cxpb=0.9, mutpb=0.1, hof_n=30, repair=None):
    random.seed(seed); np.random.seed(seed)
    nsp = NurseSchedulingProblem(penalty); tb = construir_toolbox(nsp)
    P = tb.population(n=pop); hof = tools.HallOfFame(hof_n)
    st = tools.Statistics(lambda i: i.fitness.values[0]); st.register("min", np.min); st.register("avg", np.mean)
    t0 = time.time()
    P, lb = eaElitism(P, tb, cxpb, mutpb, gen, hof, repair=repair, nsp=nsp, stats=st)
    best = hof.items[0]; c, w, n, hard = nsp.hardViolations(best)
    return dict(seed=seed, penalty=penalty, repair=bool(repair), tiempo=time.time()-t0,
                cost=best.fitness.values[0], hard=hard, soft=nsp.softViolations(best),
                cons=c, week=w, cover=n, factible=(hard == 0),
                mins=lb.select("min"), avgs=lb.select("avg"), best=list(best))
print("Motor evolutivo definido.")

Motor evolutivo definido.


## 3. Línea base: manejo por penalización

Se reproduce el enfoque orientativo: las violaciones duras se penalizan con un factor que degrada
el costo pero no impide explorar soluciones parcialmente inviables. Una corrida representativa
alcanza factibilidad (cero violaciones duras) y deja como costo residual las violaciones blandas
(preferencias no satisfechas).

In [ ]:
base_res = correr(penalty=5, seed=SEMILLA_BASE, pop=300, gen=200)
print(f"Costo mejor solución : {base_res['cost']:.0f}")
print(f"Violaciones duras    : {base_res['hard']}  (consecutivos={base_res['cons']}, "
      f"semana={base_res['week']}, cobertura={base_res['cover']})")
print(f"Violaciones blandas  : {base_res['soft']}   ->  factible: {base_res['factible']}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(base_res["mins"], color="C3", label="mínimo")
ax.plot(base_res["avgs"], color="C2", label="promedio")
ax.set_xlabel("Generación"); ax.set_ylabel("Costo (penalizado)")
ax.set_title("Convergencia - línea base (penalización)"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_convergencia_baseline.png"), dpi=150)
plt.close(fig); print("Guardado: fig_convergencia_baseline.png")

Costo mejor solución : 8
Violaciones duras    : 0  (consecutivos=0, semana=0, cobertura=0)
Violaciones blandas  : 8   ->  factible: True


Guardado: fig_convergencia_baseline.png


## 4. Calibración de la penalización

El factor de penalización es el punto delicado del enfoque: si es demasiado bajo, la mejora de
las preferencias blandas puede compensar una violación dura y el AG entrega soluciones inviables;
si es demasiado alto, se aplasta el gradiente que guía la optimización de las preferencias y se
pierde diversidad útil. Se barre el factor sobre varias semillas y se reportan la tasa de
factibilidad y el costo blando de las soluciones factibles.

In [ ]:
penalidades = [0, 1, 2, 5, 10, 50]
filas = []
for pen in penalidades:
    facts, hards, softs = [], [], []
    for s in range(SEMILLA_BASE, SEMILLA_BASE + 5):
        r = correr(pen, s, pop=300, gen=150)
        facts.append(r["factible"]); hards.append(r["hard"])
        if r["factible"]: softs.append(r["soft"])
    filas.append({"Penalización": pen, "Factibles": f"{sum(facts)}/5",
                  "Violac. duras (media)": f"{np.mean(hards):.1f}",
                  "Costo blando (factibles)": f"{np.mean(softs):.1f}" if softs else "—"})
tabla_pen = pd.DataFrame(filas)
print(tabla_pen.to_string(index=False))
tabla_pen.to_csv(os.path.join(FIGS_DIR, "tabla_penalizacion_exp5.csv"), index=False)

xs = penalidades
fact = [int(f["Factibles"].split("/")[0]) for f in filas]
soft = [float(f["Costo blando (factibles)"]) if f["Costo blando (factibles)"] != "—" else np.nan for f in filas]
fig, ax1 = plt.subplots(figsize=(7, 4.2))
ax1.plot(xs, fact, "o-", color="C0"); ax1.set_xscale("symlog")
ax1.set_xlabel("Factor de penalización (symlog)"); ax1.set_ylabel("Corridas factibles (de 5)", color="C0")
ax1.set_ylim(-0.3, 5.3); ax1.grid(alpha=0.3)
ax2 = ax1.twinx(); ax2.plot(xs, soft, "s--", color="C3")
ax2.set_ylabel("Costo blando (factibles)", color="C3")
ax1.set_title("Efecto del factor de penalización")
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_penalizacion_sweep.png"), dpi=150)
plt.close(fig); print("Guardado: fig_penalizacion_sweep.png")

 Penalización Factibles Violac. duras (media) Costo blando (factibles)
            0       0/5                  40.8                        —
            1       2/5                   1.0                      0.5
            2       5/5                   0.0                      6.6
            5       5/5                   0.0                      7.8
           10       5/5                   0.0                      7.2
           50       5/5                   0.0                      9.0


Guardado: fig_penalizacion_sweep.png


## 5. Mejora: manejo por reparación

Entre las estrategias posibles (penalización, descarte, reparación o una representación que elimine
las violaciones por construcción), se implementa la **reparación**: tras el cruce y la mutación,
cada individuo se corrige para satisfacer las restricciones duras —tope semanal, turnos
consecutivos y cobertura—, priorizando en el proceso las preferencias de cada enfermera. Así el AG
explora siempre sobre soluciones factibles y concentra su esfuerzo en optimizar lo blando.

In [ ]:
def reparar(nsp, schedule, passes=8):
    E = len(nsp.nurses); SPW = nsp.shiftsPerWeek; spd = nsp.shiftPerDay
    M = np.array(schedule, dtype=int).reshape(E, SPW); pref = np.array(nsp.shiftPreference)
    for _ in range(passes):
        # 1) tope de turnos por semana (quita primero turnos no preferidos)
        for e in range(E):
            ones = np.where(M[e] == 1)[0]
            if len(ones) > nsp.maxShiftsPerWeek:
                dis = [j for j in ones if pref[e][j % spd] == 0]
                for j in (dis + list(ones))[:len(ones) - nsp.maxShiftsPerWeek]: M[e][j] = 0
        # 2) turnos consecutivos
        for e in range(E):
            for j in range(SPW - 1):
                if M[e][j] == 1 and M[e][j+1] == 1: M[e][j+1] = 0
        # 3) cobertura por turno
        for j in range(SPW):
            s = j % spd; cnt = M[:, j].sum()
            if cnt > nsp.shiftMax[s]:
                asign = sorted(np.where(M[:, j] == 1)[0], key=lambda e: (pref[e][s] == 1))
                for e in asign[:cnt - nsp.shiftMax[s]]: M[e][j] = 0
            elif cnt < nsp.shiftMin[s]:
                cand = [e for e in range(E) if M[e][j] == 0
                        and (j == 0 or M[e][j-1] == 0) and (j == SPW-1 or M[e][j+1] == 0)
                        and M[e].sum() < nsp.maxShiftsPerWeek]
                cand.sort(key=lambda e: (pref[e][s] == 0))
                for e in cand[:nsp.shiftMin[s] - cnt]: M[e][j] = 1
    schedule[:] = M.reshape(-1).tolist(); return schedule

rep_res = correr(penalty=5, seed=SEMILLA_BASE, pop=200, gen=40, repair=reparar)
print(f"Con reparación -> costo={rep_res['cost']:.0f} | duras={rep_res['hard']} | "
      f"blandas={rep_res['soft']} | factible={rep_res['factible']} | t={rep_res['tiempo']:.1f}s")

Con reparación -> costo=0 | duras=0 | blandas=0 | factible=True | t=4.8s


## 6. Comparación de estrategias

Se contrastan penalización y reparación sobre 20 corridas independientes, midiendo la tasa de
factibilidad, el costo blando de las soluciones factibles y el tiempo por corrida.

In [ ]:
def resumen(strategy, penalty, pop, gen, repair, R=20):
    facts, softs, tiempos, curvas = [], [], [], []
    for s in range(SEMILLA_BASE, SEMILLA_BASE + R):
        r = correr(penalty, s, pop=pop, gen=gen, repair=repair)
        facts.append(r["factible"]); tiempos.append(r["tiempo"]); curvas.append(r["mins"])
        if r["factible"]: softs.append(r["soft"])
    return dict(strategy=strategy, factibles=sum(facts), R=R,
                soft=np.mean(softs) if softs else np.nan, tiempo=np.mean(tiempos), curva=curvas[0])

pen20 = resumen("Penalización", 5, 300, 150, None)
rep20 = resumen("Reparación",   5, 200, 40, reparar)
tabla_cmp = pd.DataFrame([
    {"Estrategia": d["strategy"], "Factibles (20 corridas)": f"{d['factibles']}/{d['R']}",
     "Costo blando medio": f"{d['soft']:.1f}", "Tiempo medio (s)": f"{d['tiempo']:.1f}"}
    for d in (pen20, rep20)])
print(tabla_cmp.to_string(index=False))
tabla_cmp.to_csv(os.path.join(FIGS_DIR, "tabla_comparacion_exp5.csv"), index=False)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(pen20["curva"], color="C3", label="Penalización (semilla base)")
ax.plot(rep20["curva"], color="C0", label="Reparación (semilla base)")
ax.set_xlabel("Generación"); ax.set_ylabel("Mejor costo penalizado")
ax.set_title("Convergencia: penalización vs. reparación"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_comparacion_estrategias.png"), dpi=150)
plt.close(fig); print("Guardado: fig_comparacion_estrategias.png")

  Estrategia Factibles (20 corridas) Costo blando medio Tiempo medio (s)
Penalización                   20/20                8.2              2.8
  Reparación                   20/20                0.0              4.8
Guardado: fig_comparacion_estrategias.png


## 7. Mejor horario obtenido

Se visualiza el mejor horario (estrategia de reparación) como una grilla enfermera × turno; cada
celda marca si la enfermera trabaja ese turno. Al ser factible, respeta todas las restricciones
duras.

In [ ]:
nsp_v = NurseSchedulingProblem(5)
best = rep_res["best"]
M = np.array(best).reshape(len(nsp_v.nurses), nsp_v.shiftsPerWeek)
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]; turnos = ["M", "T", "N"]
cols = [f"{d}\n{t}" for d in dias for t in turnos]

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.imshow(M, cmap=ListedColormap(["#f0f0f0", "#2c7fb8"]), aspect="auto")
ax.set_xticks(range(21)); ax.set_xticklabels(cols, fontsize=7)
ax.set_yticks(range(len(nsp_v.nurses))); ax.set_yticklabels([f"Enf. {n}" for n in nsp_v.nurses])
for j in range(0, 21, 3): ax.axvline(j - 0.5, color="k", lw=0.8)
ax.set_title(f"Mejor horario (reparación) - costo {rep_res['cost']:.0f}, factible={rep_res['factible']}")
fig.tight_layout(); fig.savefig(os.path.join(FIGS_DIR, "fig_horario_best.png"), dpi=150)
plt.close(fig); print("Guardado: fig_horario_best.png")

c, w, n, hard = nsp_v.hardViolations(best)
print(f"Verificación del horario -> duras={hard} (cons={c}, semana={w}, cobertura={n}), "
      f"blandas={nsp_v.softViolations(best)}")

Guardado: fig_horario_best.png
Verificación del horario -> duras=0 (cons=0, semana=0, cobertura=0), blandas=0


## 8. Escalabilidad

Se estima cómo crecen el espacio de búsqueda y el costo de evaluación al pasar del caso base
(8 enfermeras, 3 turnos, 1 semana) a un escenario mayor (30 enfermeras, 5 turnos, 4 semanas).

In [ ]:
def escenario(nombre, E, turnos_dia, semanas):
    slots = turnos_dia * 7 * semanas; bits = E * slots
    return {"Escenario": nombre, "Enfermeras": E, "Turnos/día": turnos_dia, "Semanas": semanas,
            "Bits (E×slots)": bits, "Espacio (2^bits)": f"1e{bits*np.log10(2):.0f}",
            "Costo relativo eval.": bits}

esc = [escenario("Base", 8, 3, 1), escenario("Ampliado", 30, 5, 4)]
tabla_esc = pd.DataFrame(esc)
base_bits = esc[0]["Bits (E×slots)"]
tabla_esc["Costo relativo eval."] = [f"{b/base_bits:.0f}×" for b in tabla_esc["Costo relativo eval."]]
print(tabla_esc.to_string(index=False))
tabla_esc.to_csv(os.path.join(FIGS_DIR, "tabla_escalabilidad_exp5.csv"), index=False)
print(f"\nEl espacio pasa de 2^{esc[0]['Bits (E×slots)']} a 2^{esc[1]['Bits (E×slots)']} estados; "
      f"la evaluación del fitness crece ~{esc[1]['Bits (E×slots)']//base_bits}× por individuo.")

Escenario  Enfermeras  Turnos/día  Semanas  Bits (E×slots) Espacio (2^bits) Costo relativo eval.
     Base           8           3        1             168             1e51                   1×
 Ampliado          30           5        4            4200           1e1264                  25×

El espacio pasa de 2^168 a 2^4200 estados; la evaluación del fitness crece ~25× por individuo.


## 9. Exportación de figuras para el informe

In [ ]:
zip_path = "figuras_informe_Exp5.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for nombre in sorted(os.listdir(FIGS_DIR)):
        zf.write(os.path.join(FIGS_DIR, nombre), arcname=nombre)
print("Contenido del ZIP:")
with zipfile.ZipFile(zip_path) as zf:
    for n in zf.namelist(): print("  -", n)
try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except Exception:
    print("\nFuera de Colab: ZIP en", os.path.abspath(zip_path))

Contenido del ZIP:
  - fig_comparacion_estrategias.png
  - fig_convergencia_baseline.png
  - fig_horario_best.png
  - fig_penalizacion_sweep.png
  - tabla_comparacion_exp5.csv
  - tabla_escalabilidad_exp5.csv
  - tabla_penalizacion_exp5.csv

Fuera de Colab: ZIP en /tmp/figuras_informe_Exp5.zip
